In [1]:
# ======================================
# Feature Engineering V3 - Setup
# ======================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

FEATURE_FILE_V2 = PROCESSED_DIR / "features_v2.parquet"

df = pd.read_parquet(FEATURE_FILE_V2)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

print("Loaded V2 features:", df.shape)
df.head()

Loaded V2 features: (1903828, 9)


,date,ticker,ret_6m,ret_12m,vol_12m,drawdown,pe_ratio,earnings_yield,sector
0,2010-01-04,A,NaN,NaN,NaN,0.000000,26.902868,0.037171,Healthcare
1,2010-01-05,A,NaN,NaN,NaN,-0.010863,26.902868,0.037171,Healthcare
2,2010-01-06,A,NaN,NaN,NaN,-0.014377,26.902868,0.037171,Healthcare
3,2010-01-07,A,NaN,NaN,NaN,-0.015655,26.902868,0.037171,Healthcare
4,2010-01-08,A,NaN,NaN,NaN,-0.015974,26.902868,0.037171,Healthcare


In [2]:
(df.notnull().sum() / len(df) * 100)

date              100.000000
ticker            100.000000
ret_6m             96.675225
ret_12m            93.354442
vol_12m            93.354442
drawdown          100.000000
pe_ratio           94.588114
earnings_yield    100.000000
sector             99.788689
dtype: float64

In [3]:
# ======================================
# Cross-Sectional Features (CORRECTED)
# ======================================

# 1. Rank the standard momentum/vol features
df["ret_6m_rank"] = df.groupby("date")["ret_6m"].rank(pct=True)
df["ret_12m_rank"] = df.groupby("date")["ret_12m"].rank(pct=True)
df["vol_12m_rank"] = df.groupby("date")["vol_12m"].rank(pct=True)
df["drawdown_rank"] = df.groupby("date")["drawdown"].rank(pct=True)

# 2. Rank the fundamental features
df["ey_rank"] = df.groupby("date")["earnings_yield"].rank(pct=True)

# 3. Invert volatility (lower vol = higher rank)
df["low_vol_rank"] = 1.0 - df["vol_12m_rank"]

# 4. Create the Composite Feature (Rank + Rank)
# This results in a beautifully distributed score from 0.0 to 2.0
df["quality_value_combo"] = df["ey_rank"] + df["low_vol_rank"]

print("Added corrected rank features")

Added corrected rank features


In [4]:
(df.notnull().sum() / len(df) * 100)

date                   100.000000
ticker                 100.000000
ret_6m                  96.675225
ret_12m                 93.354442
vol_12m                 93.354442
drawdown               100.000000
pe_ratio                94.588114
earnings_yield         100.000000
sector                  99.788689
ret_6m_rank             96.675225
ret_12m_rank            93.354442
vol_12m_rank            93.354442
drawdown_rank          100.000000
ey_rank                100.000000
low_vol_rank            93.354442
quality_value_combo     93.354442
dtype: float64

In [5]:
# ======================================
# Risk-Adjusted Momentum
# ======================================

df["momentum_vol_adj"] = df["ret_12m"] / (df["vol_12m"] + 1e-6)

print("Added momentum_vol_adj")

Added momentum_vol_adj


In [6]:
(df.notnull().sum() / len(df) * 100)

date                   100.000000
ticker                 100.000000
ret_6m                  96.675225
ret_12m                 93.354442
vol_12m                 93.354442
drawdown               100.000000
pe_ratio                94.588114
earnings_yield         100.000000
sector                  99.788689
ret_6m_rank             96.675225
ret_12m_rank            93.354442
vol_12m_rank            93.354442
drawdown_rank          100.000000
ey_rank                100.000000
low_vol_rank            93.354442
quality_value_combo     93.354442
momentum_vol_adj        93.354442
dtype: float64

In [7]:
# ======================================
# Simple Composite Signals
# ======================================

df["momentum_composite"] = (
    df["ret_6m_rank"] + df["ret_12m_rank"]
) / 2


print("Added composite features")

Added composite features


In [8]:
(df.notnull().sum() / len(df) * 100)

date                   100.000000
ticker                 100.000000
ret_6m                  96.675225
ret_12m                 93.354442
vol_12m                 93.354442
drawdown               100.000000
pe_ratio                94.588114
earnings_yield         100.000000
sector                  99.788689
ret_6m_rank             96.675225
ret_12m_rank            93.354442
vol_12m_rank            93.354442
drawdown_rank          100.000000
ey_rank                100.000000
low_vol_rank            93.354442
quality_value_combo     93.354442
momentum_vol_adj        93.354442
momentum_composite      93.354442
dtype: float64

In [9]:
# ======================================
# Clean Features
# ======================================

df = df.replace([np.inf, -np.inf], np.nan)

# Fill any new feature NaNs
rank_cols = [
    "ret_6m_rank", "ret_12m_rank",
    "vol_12m_rank", "drawdown_rank",
    "low_vol_rank"
]

df[rank_cols] = df[rank_cols].fillna(0.5)  # neutral rank

df["momentum_vol_adj"] = df["momentum_vol_adj"].fillna(0)

print("Cleaned V3 features")

Cleaned V3 features


In [10]:
df.notnull().sum() / len(df) * 100

date                   100.000000
ticker                 100.000000
ret_6m                  96.675225
ret_12m                 93.354442
vol_12m                 93.354442
drawdown               100.000000
pe_ratio                94.588114
earnings_yield         100.000000
sector                  99.788689
ret_6m_rank            100.000000
ret_12m_rank           100.000000
vol_12m_rank           100.000000
drawdown_rank          100.000000
ey_rank                100.000000
low_vol_rank           100.000000
quality_value_combo     93.354442
momentum_vol_adj       100.000000
momentum_composite      93.354442
dtype: float64

In [11]:
# ======================================
# Save Features V3
# ======================================

FEATURE_FILE_V3 = PROCESSED_DIR / "features_v3.parquet"

feature_cols = [
    "date",
    "ticker",

    # V1
    "ret_6m",
    "ret_12m",
    "vol_12m",
    "drawdown",

    # V2
    "pe_ratio",
    "earnings_yield",

    # V3 (NEW)
    "ret_6m_rank",
    "ret_12m_rank",
    "vol_12m_rank",
    "drawdown_rank",
    "low_vol_rank",
    "momentum_vol_adj",
    "momentum_composite",
    "quality_value_combo",

    # metadata
    "sector"
]

features = df[feature_cols].copy()

features.to_parquet(FEATURE_FILE_V3, index=False)

print(f"Saved V3 features to: {FEATURE_FILE_V3}")
print(features.head())

Saved V3 features to: /Users/neilyejjey/stock_signal_engine_v1/data/processed/features_v3.parquet
        date ticker  ret_6m  ret_12m  vol_12m  drawdown   pe_ratio  \
0 2010-01-04      A     NaN      NaN      NaN  0.000000  26.902868   
1 2010-01-05      A     NaN      NaN      NaN -0.010863  26.902868   
2 2010-01-06      A     NaN      NaN      NaN -0.014377  26.902868   
3 2010-01-07      A     NaN      NaN      NaN -0.015655  26.902868   
4 2010-01-08      A     NaN      NaN      NaN -0.015974  26.902868   

   earnings_yield  ret_6m_rank  ret_12m_rank  vol_12m_rank  drawdown_rank  \
0        0.037171          0.5           0.5           0.5       0.501182   
1        0.037171          0.5           0.5           0.5       0.193396   
2        0.037171          0.5           0.5           0.5       0.212264   
3        0.037171          0.5           0.5           0.5       0.252358   
4        0.037171          0.5           0.5           0.5       0.306604   

   low_vol_rank  m

In [12]:
features.notnull().sum() / len(features) * 100

date                   100.000000
ticker                 100.000000
ret_6m                  96.675225
ret_12m                 93.354442
vol_12m                 93.354442
drawdown               100.000000
pe_ratio                94.588114
earnings_yield         100.000000
ret_6m_rank            100.000000
ret_12m_rank           100.000000
vol_12m_rank           100.000000
drawdown_rank          100.000000
low_vol_rank           100.000000
momentum_vol_adj       100.000000
momentum_composite      93.354442
quality_value_combo     93.354442
sector                  99.788689
dtype: float64

In [13]:
features

,date,ticker,ret_6m,ret_12m,vol_12m,drawdown,pe_ratio,earnings_yield,ret_6m_rank,ret_12m_rank,vol_12m_rank,drawdown_rank,low_vol_rank,momentum_vol_adj,momentum_composite,quality_value_combo,sector
0,2010-01-04,A,NaN,NaN,NaN,0.000000,26.902868,0.037171,0.500000,0.500000,0.500000,0.501182,0.500000,0.000000,NaN,NaN,Healthcare
1,2010-01-05,A,NaN,NaN,NaN,-0.010863,26.902868,0.037171,0.500000,0.500000,0.500000,0.193396,0.500000,0.000000,NaN,NaN,Healthcare
2,2010-01-06,A,NaN,NaN,NaN,-0.014377,26.902868,0.037171,0.500000,0.500000,0.500000,0.212264,0.500000,0.000000,NaN,NaN,Healthcare
3,2010-01-07,A,NaN,NaN,NaN,-0.015655,26.902868,0.037171,0.500000,0.500000,0.500000,0.252358,0.500000,0.000000,NaN,NaN,Healthcare
4,2010-01-08,A,NaN,NaN,NaN,-0.015974,26.902868,0.037171,0.500000,0.500000,0.500000,0.306604,0.500000,0.000000,NaN,NaN,Healthcare
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1903823,2025-12-24,ZTS,-0.188985,-0.228600,0.018299,-0.471565,20.328903,0.049191,0.061753,0.079840,0.479042,0.123260,0.520958,-12.491890,0.070797,1.240640,Healthcare
1903824,2025-12-26,ZTS,-0.186188,-0.221406,0.018303,-0.468449,20.328903,0.049191,0.061753,0.081836,0.479042,0.125249,0.520958,-12.095988,0.071795,1.240640,Healthcare
1903825,2025-12-29,ZTS,-0.186653,-0.224929,0.018302,-0.469502,20.328903,0.049191,0.067729,0.085828,0.479042,0.125249,0.520958,-12.289231,0.076779,1.240640,Healthcare
1903826,2025-12-30,ZTS,-0.200488,-0.226137,0.018300,-0.467691,20.328903,0.049191,0.065737,0.085828,0.479042,0.121272,0.520958,-12.356356,0.075783,1.240640,Healthcare
